# ĐỀ TÀI NCKH: XÂY DỰNG ỨNG DỤNG CHATBOT HỖ TRỢ HỌC SINH TIỂU HỌC BỔ SUNG KIẾN THỨC TOÁN HỌC
- Mã số đề tài: CNTT-2026-35
- Mục tiêu: Thử nghiệm suy luận mô hình tinh chỉnh (Fine-tuned LLM) với các chế độ sư phạm tiểu học: Gợi ý từng bước (Socratic), Phương pháp sơ đồ trực quan và Giải chi tiết chuẩn GD&ĐT.

In [1]:
%%capture
import os

# Cài đặt unsloth và unsloth_zoo tương thích Kaggle
!pip install --no-deps unsloth_zoo unsloth
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps bitsandbytes accelerate peft trl triton
!pip install transformers==4.56.2 ipywidgets

In [2]:
from unsloth import FastLanguageModel
import os
import re
import torch

# Đường dẫn tới thư mục checkpoint sau khi bạn Add Dataset trên Kaggle
# Cấu trúc Kaggle: /kaggle/input/<tên-dataset-của-bạn>/checkpoint-30
saved_model_path = "/kaggle/input/models/aegold/math-nckh/transformers/default/1/checkpoint-30"

if not os.path.exists(saved_model_path):
    # Đường dẫn dự phòng nếu thư mục nằm ở root của dataset
    saved_model_path = "/kaggle/input/models/aegold/math-nckh/transformers/default/1"

print(f"Đang tải mô hình từ: {saved_model_path}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = saved_model_path,
    max_seq_length = 4096,   # Mở rộng chiều dài ngữ cảnh lên 4096 để không bị cutoff
    load_in_4bit = True
)

print("✅ Đã nạp thành công Base Model và Adapter LoRA!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Đang tải mô hình từ: /kaggle/input/models/aegold/math-nckh/transformers/default/1/checkpoint-30
==((====))==  Unsloth 2026.9.2: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 6.008 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 9.804 GiB requested
  cuda:0  budget  12.98 GiB  weights  3.141 GiB  free  9.837 GiB  reserve  9.804 GiB
  cuda:1  budget  13.00 GiB  weights  2.866 GiB

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[unsloth.import_fixes|WARNING]Unsloth: Ignoring an unusable torchao so LoRA can still be built (Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported). Run `pip install --upgrade torchao` if you need torchao quantization.
Unsloth 2026.9.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ Đã nạp thành công Base Model và Adapter LoRA!


In [3]:
import re

# Hệ thống Prompts sư phạm tiểu học chuẩn hóa
PROMPT_MODES = {
    "1. Gia sư gợi mở từng bước (Socratic cụ thể)": (
        "Bạn là một gia sư dạy Toán tiểu học tận tình, nói năng cụ thể và rõ ràng vào bài toán.\n"
        "QUY TẮC SƯ PHẠM BẮT BUỘC:\n"
        "1. Xưng hô: 'thầy/cô' - 'em'. Ngôn ngữ nhẹ nhàng, thân thiện, dễ hiểu.\n"
        "2. CẤM TUYỆT ĐỐI: Không dùng từ 'phương trình', 'hai vế', 'chuyển vế'.\n"
        "3. KHÔNG ĐƯỢC NÓI CHUNG CHUNG / MÔNG LUNG:\n"
        "   - CẤM các câu hỏi sáo rỗng như: 'Em có thể làm như thế nào?', 'Em làm gì tiếp theo?', 'Em hãy làm cẩn thận nhé?'.\n"
        "   - MỌI CÂU HỎI GỢI Ý BẮT BUỘC PHẢI GẮN VỚI CÁC CON SỐ VÀ DỮ KIỆN CỦA ĐỀ BÀI.\n"
        "4. KHI HỌC SINH MỚI ĐƯA ĐỀ BÀI (Chưa đưa ra cách làm):\n"
        "   - CẤM nói 'Thầy khen em làm đúng/làm được rồi'.\n"
        "   - Chỉ ra thành phần cần tính trước gắn với số liệu cụ thể và hỏi ĐÚNG 1 CÂU để học sinh xác định vai trò của nó.\n"
        "5. KHI HỌC SINH ĐÃ TRẢ LỜI ĐƯỢC 1 BƯỚC:\n"
        "   - Xác nhận đúng/sai ngắn gọn.\n"
        "   - Hướng dẫn bước kế tiếp bằng cách nhắc quy tắc tiểu học và gọi tên trực tiếp các con số để học sinh tính.\n"
        "6. KHÔNG tính hộ ra đáp số cuối cùng."
    ),
    "2. Hướng dẫn trực quan (Sơ đồ đoạn thẳng)": (
        "Bạn là giáo viên hướng dẫn giải toán bằng sơ đồ đoạn thẳng tiểu học.\n"
        "Nhiệm vụ: Hãy mô tả chi tiết: số lớn vẽ mấy phần, số bé vẽ mấy phần bằng nhau, "
        "tổng hoặc hiệu gắn vào đâu và đặt câu hỏi để học sinh tính số phần bằng nhau."
    ),
    "3. Bài giải mẫu chuẩn Bộ GD&ĐT (Đầy đủ)": (
        "Bạn là chuyên gia sư phạm Toán tiểu học.\n"
        "Nhiệm vụ: Trình bày bài giải mẫu hoàn chỉnh theo đúng chuẩn tiểu học: "
        "có câu lời giải rõ ràng, phép tính có kèm đơn vị trong ngoặc và đáp số rõ ràng."
    )
}

PROMPT_MODES = {
    "1. Gia sư gợi mở từng bước (Socratic cụ thể)": (
        "Bạn là một gia sư dạy Toán tiểu học tận tình, kiên nhẫn.\n"
        "QUY ĐỊNH BẮT BUỘC:\n"
        "- BẮT BUỘC DÙNG 100% TIẾNG VIỆT. TUYỆT ĐỐI KHÔNG SUY NGHĨ HOẶC TRẢ LỜI BẰNG TIẾNG ANH (No English allowed).\n"
        "- Xưng hô: 'thầy/cô' - 'em'.\n"
        "- CẤM các từ: 'phương trình', 'hai vế', 'chuyển vế'.\n"
        "- KHI HỌC SINH MỚI ĐƯA ĐỀ BÀI: Chỉ ra nhóm chưa biết (ví dụ: coi x * 3 là nhóm chưa biết) và hỏi học sinh xác định vai trò của nó trong phép tính.\n"
        "- KHI HỌC SINH ĐÃ TRẢ LỜI: Khen ngợi ngắn gọn, nhắc quy tắc tiểu học và gọi tên con số để học sinh tự tính.\n"
        "- TUYỆT ĐỐI KHÔNG giải hộ ra đáp số cuối cùng."
    ),
    "2. Hướng dẫn trực quan (Sơ đồ đoạn thẳng)": (
        "Bạn là giáo viên hướng dẫn giải toán bằng sơ đồ đoạn thẳng tiểu học bằng TIẾNG VIỆT.\n"
        "Mô tả chi tiết các phần bằng nhau và đặt câu hỏi để học sinh tự tính."
    ),
    "3. Bài giải mẫu chuẩn Bộ GD&ĐT (Đầy đủ)": (
        "Trình bày bài giải mẫu hoàn chỉnh bằng TIẾNG VIỆT theo chuẩn tiểu học: "
        "lời giải rõ ràng, phép tính có đơn vị trong ngoặc và đáp số."
    )
}

def clean_model_output(raw_text):
    """Lọc thẻ <think> và dọn sạch các đoạn nhẩm tiếng Anh nếu mô hình lỡ sinh ra"""
    # 1. Xóa nội dung trong thẻ <think>...</think>
    cleaned = re.sub(r'<think>.*?</think>', '', raw_text, flags=re.DOTALL)
    cleaned = re.sub(r'^.*?<\/think>', '', cleaned, flags=re.DOTALL) # Trường hợp mất thẻ mở nhưng còn thẻ đóng
    
    # 2. Xóa các thẻ điều khiển đặc biệt
    cleaned = cleaned.replace('<|im_end|>', '').replace('<think>', '').replace('</think>', '').strip()
    
    # 3. Chặn trường hợp mô hình nhẩm tiếng Anh (bắt đầu bằng các từ nhẩm kinh điển của Qwen)
    english_thought_starters = [
        "okay, let's see", "first, let's", "the student is", "the problem is",
        "let me see", "we need to solve", "hmm, how do i", "starting with"
    ]
    
    # Nếu kết quả bắt đầu bằng tiếng Anh, tách lấy phần tiếng Việt phía sau (nếu có)
    lower_text = cleaned.lower()
    for starter in english_thought_starters:
        if lower_text.startswith(starter):
            # Tìm vị trí xuất hiện câu tiếng Việt đầu tiên (thường có dấu hoặc từ xưng hô)
            vietnamese_match = re.search(r'(Chào em|Thầy|Cô|Trong phép|Để tìm|Bài toán|Trước tiên)', cleaned, re.IGNORECASE)
            if vietnamese_match:
                cleaned = cleaned[vietnamese_match.start():].strip()
            else:
                # Nếu toàn bộ là tiếng Anh nhẩm, đưa ra câu phản hồi mặc định chuẩn sư phạm
                cleaned = "Chào em! Trong bài toán này, em hãy chú ý đến phép tính có chứa x trước nhé. Em hãy cho thầy biết phép tính này gồm những thành phần nào?"
            break

    return cleaned.strip()

def run_pedagogical_inference(user_text, system_prompt, enable_thinking=False, max_new_tokens=400, temp=0.25):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temp,
        top_p=0.85,
        repetition_penalty=1.15,
        do_sample=True
    )

    gen_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    raw_res = tokenizer.decode(gen_tokens, skip_special_tokens=True)
    return clean_model_output(raw_res)

In [4]:
# [MỤC 3 ĐỀ TÀI] Mô phỏng hội thoại đa lượt không bị nói vague
class MultiTurnMathSession:
    def __init__(self):
        self.history = [{"role": "system", "content": PROMPT_MODES["1. Gia sư gợi mở từng bước (Socratic cụ thể)"]}]

    def send_message(self, user_text):
        self.history.append({"role": "user", "content": user_text})
        
        prompt = tokenizer.apply_chat_template(
            self.history,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        out = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.25,        # Nhiệt độ thấp để bám chặt quy tắc và dữ liệu số
            top_p=0.85,
            repetition_penalty=1.15,
            do_sample=True
        )
        
        raw_res = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        bot_res = clean_model_output(raw_res)
        
        self.history.append({"role": "assistant", "content": bot_res})
        return bot_res

# Chạy thử nghiệm kiểm chứng
session = MultiTurnMathSession()

print("👦 Học sinh (Lượt 1): Em không biết làm bài này: Tìm x, biết: x * 3 - 4 = 11")
print(f"🤖 Gia sư AI:\n{session.send_message('Em không biết làm bài này: Tìm x, biết: x * 3 - 4 = 11')}\n")

print("👦 Học sinh (Lượt 2): Trong phép trừ thì (x * 3) là số bị trừ đúng không ạ?")
print(f"🤖 Gia sư AI:\n{session.send_message('Trong phép trừ thì (x * 3) là số bị trừ đúng không ạ?')}\n")

print("👦 Học sinh (Lượt 3): Số bị trừ bằng 11 + 4 = 15. Vậy x * 3 = 15 rồi thì tìm x thế nào nữa ạ?")
print(f"🤖 Gia sư AI:\n{session.send_message('Số bị trừ bằng 11 + 4 = 15. Vậy x * 3 = 15 rồi thì tìm x thế nào nữa ạ?')}\n")

👦 Học sinh (Lượt 1): Em không biết làm bài này: Tìm x, biết: x * 3 - 4 = 11
🤖 Gia sư AI:
Thầy thấy em đang gặp khó khăn với bài toán tìm x. Thầy sẽ giúp em từng bước nhé.

Trước tiên, thầy cần em xác định nhóm nào chưa biết trong phép tính x * 3 - 4 = 11. Theo thầy, nhóm chưa biết ở đây có thể là x hoặc kết quả của phép nhân x * 3. Em hãy cho thầy biết nhóm nào em nghĩ là chưa biết?

👦 Học sinh (Lượt 2): Trong phép trừ thì (x * 3) là số bị trừ đúng không ạ?
🤖 Gia sư AI:
Thầy rất vui vì em đã nhận ra rằng trong phép trừ, (x * 3) chính là số bị trừ. Đây là bước quan trọng để giải bài toán. Tiếp theo, thầy muốn em xác định phần còn lại của phép trừ, tức là số trừ và hiệu. Em có thể chỉ rõ được không?

👦 Học sinh (Lượt 3): Số bị trừ bằng 11 + 4 = 15. Vậy x * 3 = 15 rồi thì tìm x thế nào nữa ạ?
🤖 Gia sư AI:
Thầy khen em rất tốt! Em đã hiểu rõ cách tìm số bị trừ trong phép trừ. Giờ em đã biết rằng x * 3 = 15. Để tìm x, em cần thực hiện phép chia 15 cho 3. Hãy thử làm như vậy đi, em có thể tí

In [5]:
# [MỤC 6 ĐỀ TÀI] Benchmark tự động đánh giá phản hồi lượt 1 theo quy tắc mới
elementary_tests = [
    {"grade": "Lớp 2", "topic": "Tìm thành phần chưa biết", "prompt": "Tìm x, biết: x + 18 = 45"},
    {"grade": "Lớp 3", "topic": "Tìm thừa số chưa biết", "prompt": "Tìm x, biết: x * 4 = 36"},
    {"grade": "Lớp 4", "topic": "Toán Tổng - Hiệu", "prompt": "Hai bạn An và Bình có tổng cộng 24 cái kẹo. An có nhiều hơn Bình 6 cái kẹo. Hỏi mỗi bạn có bao nhiêu cái kẹo?"},
    {"grade": "Lớp 5", "topic": "Toán chuyển động đều", "prompt": "Một ô tô đi quãng đường dài 120 km trong thời gian 2 giờ 30 phút. Hãy tính vận tốc của ô tô."}
]

print("="*70)
print("KẾT QUẢ ĐÁNH GIÁ PHẢN HỒI GỢI MỞ THEO CHƯƠNG TRÌNH TIỂU HỌC")
print("="*70)

for item in elementary_tests:
    response = run_pedagogical_inference(
        user_text=item["prompt"],
        system_prompt=PROMPT_MODES["1. Gia sư gợi mở từng bước (Socratic cụ thể)"],
        enable_thinking=False,
        max_new_tokens=300,
        temp=0.25
    )
    print(f"\n📚 [{item['grade']} - {item['topic']}]")
    print(f"📝 Đề bài: {item['prompt']}")
    print(f"💡 Phản hồi gợi mở:\n{response}")
    print("-" * 70)

KẾT QUẢ ĐÁNH GIÁ PHẢN HỒI GỢI MỞ THEO CHƯƠNG TRÌNH TIỂU HỌC

📚 [Lớp 2 - Tìm thành phần chưa biết]
📝 Đề bài: Tìm x, biết: x + 18 = 45
💡 Phản hồi gợi mở:
Thầy thấy em đã tìm được nhóm chưa biết rồi đấy! Nhóm đó chính là x nhé? Em hãy xác định x có vai trò gì trong phép cộng này nhé?
----------------------------------------------------------------------

📚 [Lớp 3 - Tìm thừa số chưa biết]
📝 Đề bài: Tìm x, biết: x * 4 = 36
💡 Phản hồi gợi mở:
Thầy thấy em đã tìm được nhóm chưa biết rồi đó là x. Nhóm này đang làm gì ở đây?
----------------------------------------------------------------------

📚 [Lớp 4 - Toán Tổng - Hiệu]
📝 Đề bài: Hai bạn An và Bình có tổng cộng 24 cái kẹo. An có nhiều hơn Bình 6 cái kẹo. Hỏi mỗi bạn có bao nhiêu cái kẹo?
💡 Phản hồi gợi mở:
Thầy thấy em đã hiểu rõ đề bài rồi đấy! Em hãy xác định nhóm nào chưa biết nhé. Trong bài này, nhóm chưa biết là số kẹo mà mỗi bạn có. Thầy sẽ giúp em tìm cách giải quyết bằng cách đặt biến đại diện cho số kẹo của từng bạn. Em có thể chọn

In [6]:
# @title 💬 PHÒNG HỌC TƯƠNG TÁC GIA SƯ TOÁN TIỂU HỌC (LƯU CONTEXT, CHỐNG NÓI VAGUE, CÓ NÚT RESET)
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Markdown

# --- 1. PROMPT GIA SƯ ĐẶC TRỊ GỢI Ý MÔNG LUNG (ACTIONABLE SOCRATIC) ---
CONCRETE_SOCRATIC_PROMPT = """Bạn là một gia sư dạy Toán tiểu học tận tình, nói năng cụ thể và rõ ràng vào bài toán.
QUY TẮC SƯ PHẠM BẮT BUỘC (TUYỆT ĐỐI TUÂN THỦ):
1. Xưng hô: 'thầy/cô' - 'em'. Ngôn ngữ nhẹ nhàng, thân thiện, dễ hiểu.
2. CẤM TUYỆT ĐỐI: Không dùng từ 'phương trình', 'hai vế', 'chuyển vế'.
3. KHÔNG ĐƯỢC NÓI CHUNG CHUNG / MÔNG LUNG:
   - CẤM các câu hỏi sáo rỗng vô nghĩa như: 'Em có thể làm như thế nào?', 'Em làm gì tiếp theo?', 'Em hãy làm cẩn thận nhé?'.
   - MỌI CÂU HỎI GỢI Ý BẮT BUỘC PHẢI GẮN VỚI SỐ LIỆU CỤ THỂ CỦA BÀI TOÁN.
4. KHI HỌC SINH MỚI ĐƯA ĐỀ BÀI (LƯỢT 1):
   - Tuyệt đối KHÔNG nói 'Thầy khen em làm đúng rồi' (vì học sinh chưa làm gì cả).
   - Chỉ ra thành phần cần ưu tiên tính trước gắn với con số cụ thể trong đề.
   - Ví dụ: 'Chào em! Trong bài x * 3 - 4 = 11, ta coi (x * 3) là một nhóm chưa biết. Trong phép trừ có số trừ là 4 và hiệu là 11, em hãy cho thầy biết (x * 3) đóng vai trò là số gì?'
5. KHI HỌC SINH ĐÃ TRẢ LỜI ĐƯỢC 1 BƯỚC:
   - Xác nhận đúng/sai ngắn gọn.
   - Hướng dẫn bước kế tiếp bằng cách nhắc lại quy tắc tiểu học và gọi tên trực tiếp các con số để học sinh tính.
   - Ví dụ: 'Đúng rồi em! Muốn tìm thừa số x, ta lấy tích là 15 chia cho thừa số đã biết là 3. Em hãy tính giúp thầy xem 15 chia 3 bằng mấy nhé?'
6. KHÔNG đưa ra đáp số cuối cùng cho đến khi học sinh tự tính ra."""

SYSTEM_MODES = {
    "1. Gia sư gợi mở từng bước (Socratic cụ thể)": CONCRETE_SOCRATIC_PROMPT,
    "2. Hướng dẫn trực quan (Sơ đồ đoạn thẳng)": (
        "Bạn là giáo viên hướng dẫn giải toán bằng sơ đồ đoạn thẳng tiểu học. "
        "Hãy mô tả chi tiết: số lớn vẽ mấy phần, số bé vẽ mấy phần bằng nhau, tổng/hiệu gắn vào đâu và hỏi học sinh số phần bằng nhau là bao nhiêu."
    ),
    "3. Bài giải mẫu chuẩn Bộ GD&ĐT (Đầy đủ)": (
        "Trình bày lời giải hoàn chỉnh theo đúng chuẩn tiểu học: câu lời giải rõ ràng, phép tính có đơn vị trong ngoặc và đáp số."
    )
}

# --- 2. BỘ LƯU TRỮ CONTEXT & TRẠNG THÁI ---
conversation_history = []

def init_new_session(mode_name):
    global conversation_history
    conversation_history = [
        {"role": "system", "content": SYSTEM_MODES[mode_name]}
    ]

# --- 3. THIẾT KẾ GIAO DIỆN CHAT UI ---
w_mode = widgets.Dropdown(
    options=list(SYSTEM_MODES.keys()),
    value="1. Gia sư gợi mở từng bước (Socratic cụ thể)",
    description="Chế độ:",
    style={'description_width': '100px'},
    layout=widgets.Layout(width='95%')
)

w_user_text = widgets.Text(
    value="Tìm x, biết: x * 3 - 4 = 11",
    placeholder="Nhập đề bài hoặc câu trả lời tiếp theo của em...",
    description="Tin nhắn:",
    style={'description_width': '100px'},
    layout=widgets.Layout(width='70%')
)

w_send = widgets.Button(
    description="Gửi tin nhắn 💬",
    button_style='primary',
    layout=widgets.Layout(width='14%', height='36px')
)

w_reset = widgets.Button(
    description="🔄 Làm mới hội thoại",
    button_style='warning',
    layout=widgets.Layout(width='14%', height='36px')
)

chat_display_area = widgets.Output()

# --- 4. HÀM HIỂN THỊ TOÀN BỘ LỊCH SỬ CHAT TRỰC QUAN ---
def refresh_chat_ui():
    with chat_display_area:
        clear_output()
        if len(conversation_history) <= 1:
            display(HTML("<p style='color: #5f6368; font-style: italic;'>Phiên học đang trống. Hãy nhập câu hỏi/bài toán và nhấn <b>Gửi tin nhắn</b> để bắt đầu!</p>"))
            return

        for msg in conversation_history:
            if msg["role"] == "user":
                html_user = f"""
                <div style='display: flex; justify-content: flex-end; margin-bottom: 12px;'>
                    <div style='background: #e8f0fe; border-left: 4px solid #1a73e8; border-radius: 8px; padding: 10px 14px; max-width: 80%;'>
                        <span style='font-size: 12px; font-weight: bold; color: #174ea6;'>👦 HỌC SINH</span>
                        <div style='margin-top: 4px; color: #202124; font-size: 14px; line-height: 1.5;'>{msg['content']}</div>
                    </div>
                </div>
                """
                display(HTML(html_user))
            elif msg["role"] == "assistant":
                html_bot = f"""
                <div style='display: flex; justify-content: flex-start; margin-bottom: 12px;'>
                    <div style='background: #f8f9fa; border-left: 4px solid #34a853; border-radius: 8px; padding: 10px 14px; max-width: 80%; box-shadow: 0 1px 2px rgba(0,0,0,0.05);'>
                        <span style='font-size: 12px; font-weight: bold; color: #137333;'>🤖 GIA SƯ AI</span>
                        <div style='margin-top: 4px; color: #202124; font-size: 14px; line-height: 1.5;'>
                """
                display(HTML(html_bot))
                display(Markdown(msg['content']))
                display(HTML("</div></div></div>"))

# --- 5. XỬ LÝ SỰ KIỆN GỬI TIN NHẮN & SUY LUẬN MÔ HÌNH ---
def on_send_click(b):
    global conversation_history
    text = w_user_text.value.strip()
    if not text:
        return

    # Nếu chưa khởi tạo session hoặc vừa đổi chế độ, nạp system prompt mới
    if len(conversation_history) == 0:
        init_new_session(w_mode.value)

    # Đưa tin nhắn của học sinh vào lịch sử context
    conversation_history.append({"role": "user", "content": text})
    w_user_text.value = ""
    refresh_chat_ui()

    with chat_display_area:
        display(HTML("<p style='color: #1a73e8; font-style: italic;'>⏳ Gia sư AI đang soạn câu gợi ý cụ thể...</p>"))

    # Áp dụng template với toàn bộ ngữ cảnh trò chuyện
    chat_prompt = tokenizer.apply_chat_template(
        conversation_history,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    
    inputs = tokenizer(chat_prompt, return_tensors="pt").to("cuda")

    # Sinh phản hồi ngắn gọn, tập trung
    output_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.25,        # Đặt nhiệt độ thấp để bot bám sát dữ liệu số, không bị nói lan man
        top_p=0.85,
        repetition_penalty=1.15,
        do_sample=True
    )

    gen_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    raw_response = tokenizer.decode(gen_tokens, skip_special_tokens=True)
    clean_response = clean_model_output(raw_response)

    # Lưu câu trả lời vào context memory và render lại toàn bộ màn hình
    conversation_history.append({"role": "assistant", "content": clean_response})
    refresh_chat_ui()

def on_reset_click(b):
    init_new_session(w_mode.value)
    refresh_chat_ui()

def on_mode_change(change):
    # Tự động làm mới phiên khi chuyển đổi chế độ
    init_new_session(change['new'])
    refresh_chat_ui()

w_send.on_click(on_send_click)
w_reset.on_click(on_reset_click)
w_mode.observe(on_mode_change, names='value')

# Khởi tạo mặc định
init_new_session(w_mode.value)

# --- 6. RENDER GIAO DIỆN RA MÀN HÌNH ---
toolbar = widgets.HBox([w_user_text, w_send, w_reset], layout=widgets.Layout(align_items='center'))
display(widgets.VBox([w_mode, toolbar]))
display(HTML("<hr style='margin: 10px 0;'>"))
display(chat_display_area)
refresh_chat_ui()

Output()